In [1]:
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType, IntegerType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import *
import time


spark = (
    SparkSession.builder
    .appName("Pre-partitioning")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 01:51:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/07 01:51:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# de-active broadcast joins since we are running joins on small datasets but do not want broadcast for demo
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [ ]:
# dont touch this

# addColumns(initialTable, 3) => dataframe wit columns "id", "newCol1", "newCol2", "newCol3"
def addColumns(df, n):
    new_columns = [f"id * {i} as newCol{i}" for i in range(1, n + 1)]
    return df.selectExpr("id", *new_columns)


initialTable = spark.range(1, 10000000).repartition(10)

narrowTable = spark.range(1, 5000000).repartition(7)


In [ ]:
# scenario 1
wideTable = addColumns(initialTable, 30)

join1 = wideTable.join(narrowTable, "id")


join1.count()

In [ ]:
# scenario 2
altNarrow = narrowTable.repartition()